In [13]:
import pandas as pd
from pathlib import Path
import os

In [14]:
# ------------- Paths -------------

BASE_DIR = "/data0/nksp2/skagit/skagit_2/skagit-met"
EXP_DATA_DIR = os.path.join(BASE_DIR, "experiments_2/data")
HYDRO_Q_PATH = os.path.join(EXP_DATA_DIR, "usgs_12200500_discharge.rdb")
HYDRO_H_PATH = os.path.join(EXP_DATA_DIR, "usgs_12200500_gage_height.rdb")

In [15]:
def load_usgs_rdb(filepath, param_name):
    print(f"Loading {param_name} from {filepath}...")
    try:
        df = pd.read_csv(filepath, sep='\t', comment='#')
        df = df.iloc[1:].reset_index(drop=True)
        val_cols = [c for c in df.columns if '_00' in c and not c.endswith('_cd')]
        if not val_cols: return pd.DataFrame(columns=['date', param_name])
        val_col = val_cols[0]
        df = df[['datetime', val_col]]
        df.columns = ['date', param_name]
        df['date'] = pd.to_datetime(df['date'])
        df[param_name] = pd.to_numeric(df[param_name], errors='coerce')
        return df
    except Exception as e:
        print(f"  Error loading {filepath}: {e}")
        return pd.DataFrame(columns=['date', param_name])

In [16]:
# 1. Create Base Timeline (1980-2025)
dates = pd.date_range(start="1980-01-01", end="2025-12-31", freq='D')
final_df = pd.DataFrame({'date': dates})

# 2. Load Hydrology
q_df = load_usgs_rdb(HYDRO_Q_PATH, 'discharge_cfs')
h_df = load_usgs_rdb(HYDRO_H_PATH, 'gage_height_ft')
hydro_df = pd.merge(q_df, h_df, on='date', how='outer')
final_df = pd.merge(final_df, hydro_df, on='date', how='left')

Loading discharge_cfs from /data0/nksp2/skagit/skagit_2/skagit-met/experiments_2/data/usgs_12200500_discharge.rdb...
Loading gage_height_ft from /data0/nksp2/skagit/skagit_2/skagit-met/experiments_2/data/usgs_12200500_gage_height.rdb...


In [17]:
final_df

,date,discharge_cfs,gage_height_ft
0,1980-01-01,16500,NaN
1,1980-01-02,16700,NaN
2,1980-01-03,17700,NaN
3,1980-01-04,16400,NaN
4,1980-01-05,15800,NaN
...,...,...,...
16797,2025-12-27,25900,18.24
16798,2025-12-28,22900,17.30
16799,2025-12-29,21800,16.93
16800,2025-12-30,21800,16.93


In [18]:
final_df = final_df.sort_values('discharge_cfs', ascending=False)
final_df['discharge_cfs'] = final_df['discharge_cfs'] / 35.3147
final_df.head(20)

,date,discharge_cfs,gage_height_ft
3981,1990-11-25,4020.988427,36.62
5812,1995-11-30,3737.820228,36.48
3967,1990-11-11,3681.186588,35.63
9808,2006-11-08,3539.602488,32.66
15295,2021-11-16,3369.701569,NaN
9807,2006-11-07,3284.751109,31.76
8695,2003-10-22,3256.434289,34.07
16782,2025-12-12,3171.483830,35.06
16781,2025-12-11,2888.315631,33.68
361,1980-12-27,2859.998811,NaN
